In [38]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [39]:
torch.manual_seed(42)

In [40]:
df = pd.read_csv('fmnist_small.csv')

In [41]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [42]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [43]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state = 42)

In [44]:
X_train = X_train/255.0
X_test = X_test/255.0

In [45]:
#create CustomDataset Class
class CustomDataset(Dataset):

  def __init__(self,features,labels):
    self.features = torch.tensor(features, dtype = torch.float32)
    self.labels = torch.tensor(labels, dtype = torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):

    return self.features[index], self.labels[index]


In [46]:
#create train_dataset object

train_dataset = CustomDataset(X_train,y_train)

In [47]:
#create test_dataset object
test_dataset = CustomDataset(X_test,y_test)

In [48]:
#create train and test loader
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = True)

In [49]:
#define NN class
class MyNN(nn.Module):

  def __init__(self,num_features):
    super().__init__()

    self.model = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,10)
    )

  def forward(self,x):
    return self.model(x
    )

In [50]:
#learning rate and epoch
learning_rate = 0.1
epochs = 100

In [51]:
#instantiate the model
model = MyNN(X_train.shape[1])

#loss function
criterion = nn.CrossEntropyLoss()

#optimizer
optimizer = optim.SGD(model.parameters(), lr = learning_rate)

In [54]:
#training loop

for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features,batch_labels in train_loader:
        #forward pass
        outputs = model(batch_features)
        #calculate loss
        loss = criterion(outputs,batch_labels)
        #backprop
        optimizer.zero_grad()
        loss.backward()
        #update gradients
        optimizer.step()

        total_epoch_loss += loss.item()

        avg_epoch_loss = total_epoch_loss/len(train_loader)

    print(f"Epoch: {epoch+1}, Loss: {avg_epoch_loss}")

Epoch: 1, Loss: 1.307383732398351
Epoch: 2, Loss: 0.7654244385162989
Epoch: 3, Loss: 0.6439137537280718
Epoch: 4, Loss: 0.5819168621301651
Epoch: 5, Loss: 0.5333500763773918
Epoch: 6, Loss: 0.4916381385922432
Epoch: 7, Loss: 0.46908331553141275
Epoch: 8, Loss: 0.4406186990936597
Epoch: 9, Loss: 0.422690896888574
Epoch: 10, Loss: 0.40773864487806954
Epoch: 11, Loss: 0.3853214595715205
Epoch: 12, Loss: 0.37562320719162623
Epoch: 13, Loss: 0.34652384142080944
Epoch: 14, Loss: 0.33483974501490593
Epoch: 15, Loss: 0.32256466840704284
Epoch: 16, Loss: 0.30917051682869595
Epoch: 17, Loss: 0.2932174900174141
Epoch: 18, Loss: 0.2887418281535308
Epoch: 19, Loss: 0.28020624568065006
Epoch: 20, Loss: 0.2599971701204777
Epoch: 21, Loss: 0.2605044812709093
Epoch: 22, Loss: 0.24116242033739885
Epoch: 23, Loss: 0.2385901505748431
Epoch: 24, Loss: 0.23092645451426505
Epoch: 25, Loss: 0.22588358248273532
Epoch: 26, Loss: 0.22084147882958252
Epoch: 27, Loss: 0.19894037174681822
Epoch: 28, Loss: 0.2030485

In [55]:
model.eval()

MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [58]:
#evaluation code
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(f"Accuracy: {correct/total}")

Accuracy: 0.8358333333333333
